# New procedure for VTEC data extraction

In [1]:
from scripts.new_dtec_extractor import get_dtec_data_from_stations
from scripts.new_dtec_analysis import DTECAnalysis

from scipy.spatial.distance import mahalanobis
import numpy as np
import polars as pl

In [2]:
files_path = "./data/new data/"

min_elev = 30.0
window_size = 240
poly_order = 5

data = get_dtec_data_from_stations(data_path = files_path,
                                   min_elev = min_elev,
                                   window_size = window_size,
                                   poly_order = poly_order)

/home/federico/miniconda3/envs/data-science/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/federico/miniconda3/envs/data-science/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: divide by zero encountered in divide
  ret = ret.dtype.type(ret / rcount)


In [3]:
dtec_series, mds_analysis = DTECAnalysis(k_max = 3).compute_mds_embeddings(data)

Processing stations:   0%|          | 0/50 [00:00<?, ?station/s]

Computing distance matrix for 3387 time series...


Computing MDS embeddings:   0%|          | 0/3 [00:00<?, ?k/s]

In [4]:
import plotly.express as px

px.line(
    x = list(mds_analysis.keys()),
    y = [mds_analysis[k][1] for k in mds_analysis.keys()],
    labels = {"x": "Number of dimensions (k)", "y": "Stress"},
    title = "Stress vs Number of Dimensions (k)",
    width = 800, height = 500
)

In [5]:
px.scatter_3d(
    data_frame = mds_analysis[3][0],
    x = 0, y = 1, z = 2,
    color = dtec_series["station"],
    hover_data = [dtec_series["start_epoch"], dtec_series["end_epoch"]]
)

## Mahalanobis anomaly scores

Each DTEC segment is scored by its Mahalanobis distance from the centroid of the MDS embedding. Scores are calculated independently for k = 1, 2, and 3 using the embedding covariance; a pseudoinverse handles singular or collinear dimensions. Larger distances indicate more unusual segments. The dashed line marks the empirical 99th percentile as a visual reference, not a formal significance threshold.

In [6]:
def mahalanobis_dist(embedding: np.ndarray) -> np.ndarray:
    if embedding.ndim != 2 or embedding.shape[0] < 2:
        raise ValueError("Mahalanobis scoring requires at least two segment embeddings.")

    center = embedding.mean(axis=0)
    covariance = np.atleast_2d(np.cov(embedding, rowvar=False))
    inverse_covariance = np.linalg.pinv(covariance, hermitian=True)
    return np.asarray([
        mahalanobis(segment_embedding, center, inverse_covariance)
        for segment_embedding in embedding
    ])


segment_scores = dtec_series.with_row_index("segment_id")
if "sv" not in segment_scores.columns:
    segment_satellites = [
        frame["sv"][0]
        for station_segments in data.values()
        for frame in station_segments
        if np.isfinite(frame["dtec"].cast(pl.Float64).to_numpy()).any()
    ]
    if len(segment_satellites) != segment_scores.height:
        raise ValueError("Could not align satellite labels with the DTEC segment rows.")
    segment_scores = segment_scores.with_columns(pl.Series("sv", segment_satellites))

score_columns = []
for k in (1, 2, 3):
    coordinates = np.asarray(mds_analysis[k][0], dtype=float)
    scores = mahalanobis_dist(coordinates)
    score_column = f"mahalanobis_k{k}"
    score_columns.append(score_column)
    segment_scores = segment_scores.with_columns(pl.Series(score_column, scores))

metadata_columns = [
    column for column in ("segment_id", "station", "sv", "start_epoch", "end_epoch")
    if column in segment_scores.columns
]
top_rows = []
for k, score_column in enumerate(score_columns, start=1):
    top_segments = (
        segment_scores
        .select([*metadata_columns, score_column])
        .sort(score_column, descending=True)
        .head(10)
    )
    for rank, segment in enumerate(top_segments.iter_rows(named=True), start=1):
        top_rows.append({
            "k": k,
            "rank": rank,
            **{column: segment[column] for column in metadata_columns},
            "mahalanobis_distance": segment[score_column]
        })

top_dtec_segments = pl.DataFrame(top_rows)
display(top_dtec_segments)

from plotly.subplots import make_subplots
import plotly.graph_objects as go

score_figure = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=["1D embedding", "2D embedding", "3D embedding"],
    horizontal_spacing=0.09
)
poster_colors = ["#168C83", "#3567A5", "#D9664B"]

for k, score_column in enumerate(score_columns, start=1):
    ordered_scores = np.sort(segment_scores[score_column].to_numpy())[::-1]
    score_figure.add_trace(
        go.Scatter(
            x=np.arange(1, len(ordered_scores) + 1),
            y=ordered_scores,
            mode="lines",
            line={"color": poster_colors[k - 1], "width": 3},
            name=f"k={k}",
            hovertemplate="Rank %{x}<br>Distance %{y:.3f}<extra></extra>"
        ),
        row=1,
        col=k
    )
    percentile_99 = float(np.quantile(ordered_scores, 0.99))
    score_figure.add_hline(
        y=percentile_99,
        line={"color": "#D9664B", "width": 1.5, "dash": "dash"},
        annotation_text=f"99th percentile: {percentile_99:.2f}",
        annotation_position="top left",
        row=1,
        col=k
    )
    score_figure.update_xaxes(title_text="Segment rank", row=1, col=k)
    score_figure.update_yaxes(title_text="Mahalanobis distance", row=1, col=k)

score_figure.update_layout(
    title={"text": "DTEC Segment Anomaly Scores by MDS Dimension", "x": 0.5},
    template="plotly_white",
    width=1500,
    height=620,
    font={"family": "Aptos, sans-serif", "size": 15, "color": "#202A33"},
    showlegend=False,
    margin={"l": 75, "r": 35, "t": 100, "b": 70}
)
score_figure.show()

k,rank,segment_id,station,sv,start_epoch,end_epoch,mahalanobis_distance
i64,i64,i64,str,str,datetime[μs],datetime[μs],f64
1,1,1182,"""p796""","""E04""",2025-11-10 21:43:00,2025-11-11 01:40:45,18.507211
1,2,920,"""usmx""","""G19""",2025-11-10 23:36:00,2025-11-11 04:20:00,18.244368
1,3,910,"""usmx""","""G22""",2025-11-10 22:19:30,2025-11-11 03:56:45,16.181568
1,4,995,"""usmx""","""E21""",2025-11-10 22:41:00,2025-11-11 00:01:45,15.740159
1,5,1162,"""p796""","""G14""",2025-11-10 21:25:15,2025-11-11 02:45:30,15.339909
…,…,…,…,…,…,…,…
3,6,1171,"""p796""","""G30""",2025-11-10 20:44:45,2025-11-11 01:39:15,17.296569
3,7,2076,"""lto1""","""G07""",2025-11-10 20:16:30,2025-11-11 00:51:15,17.21371
3,8,1162,"""p796""","""G14""",2025-11-10 21:25:15,2025-11-11 02:45:30,17.029652


### Raw DTEC traces for highest-scoring segments

Each panel shows all ten segments listed for that embedding dimension. The x-axis retains the original epoch datetimes.

In [ ]:
raw_segments = [
    frame
    for station_segments in data.values()
    for frame in station_segments
    if np.isfinite(frame["dtec"].cast(pl.Float64).to_numpy()).any()
]
if len(raw_segments) != segment_scores.height:
    raise ValueError("Raw DTEC segments do not align with the scored segment rows.")

for segment_id, frame in enumerate(raw_segments):
    if (
        segment_scores["station"][segment_id] != frame["station"][0]
        or segment_scores["sv"][segment_id] != frame["sv"][0]
    ):
        raise ValueError(f"Segment metadata mismatch at segment {segment_id}.")

trace_figure = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=["k = 1", "k = 2", "k = 3"],
    horizontal_spacing=0.08
)
trace_colors = px.colors.qualitative.Dark24[:10]

for k, score_column in enumerate(score_columns, start=1):
    highest_scoring = (
        segment_scores
        .sort(score_column, descending=True)
        .head(10)
    )
    for rank, segment in enumerate(highest_scoring.iter_rows(named=True), start=1):
        frame = raw_segments[int(segment["segment_id"])]
        dtec_values = frame["dtec"].cast(pl.Float64).to_numpy()
        finite_mask = np.isfinite(dtec_values)
        dtec_values = dtec_values[finite_mask]
        epochs = frame["epoch"].to_numpy()[finite_mask]

        trace_figure.add_trace(
            go.Scatter(
                x=epochs,
                y=dtec_values,
                mode="lines",
                line={"color": trace_colors[rank - 1], "width": 1.6},
                opacity=0.8,
                name=(
                    f"Rank {rank}: {segment['station']}/{segment['sv']} "
                    f"(D={segment[score_column]:.2f})"
                ),
                showlegend=False,
                hovertemplate=(
                    "%{x|%Y-%m-%d %H:%M:%S}<br>"
                    "DTEC %{y:.4f}<extra>%{fullData.name}</extra>"
                )
            ),
            row=1,
            col=k
        )
    trace_figure.update_xaxes(
        title_text="Epoch",
        tickformat="%m-%d<br>%H:%M",
        row=1,
        col=k
    )
    trace_figure.update_yaxes(title_text="", row=1, col=k)

assert len(trace_figure.data) == 30
assert all(not trace.showlegend for trace in trace_figure.data)
assert all(
    np.issubdtype(np.asarray(trace.x).dtype, np.datetime64)
    for trace in trace_figure.data
)

trace_figure.add_annotation(
    x=-0.055,
    y=0.5,
    xref="paper",
    yref="paper",
    text="DTEC",
    textangle=-90,
    showarrow=False,
    font={"family": "Aptos, sans-serif", "size": 18, "color": "#202A33"}
)
trace_figure.update_layout(
    title={"text": "Raw DTEC Traces of Top 10 Segments by MDS Dimension", "x": 0.5},
    template="plotly_white",
    width=1500,
    height=760,
    font={"family": "Aptos, sans-serif", "size": 14, "color": "#202A33"},
    showlegend=False,
    margin={"l": 105, "r": 30, "t": 100, "b": 100}
)
trace_figure.show()

In [8]:
top_dtec_segments["sv"].unique()

sv
str
"""E23"""
"""E21"""
"""G14"""
"""G17"""
"""G30"""
"""G19"""
"""G22"""
"""G07"""
"""E04"""


## Poster-ready figure exports

Export the Mahalanobis ranking and raw DTEC traces as vector SVG and high-resolution PNG sized for a 120 cm wide poster.

In [ ]:
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from pathlib import Path

poster_output = Path("output/poster")
poster_output.mkdir(parents=True, exist_ok=True)
poster_width_inches = 120 / 2.54
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "svg.fonttype": "none",
    "axes.titleweight": "bold"
})

score_export, score_axes = plt.subplots(
    1,
    3,
    figsize=(poster_width_inches, 24),
    layout="constrained"
)
for k, (axis, score_column, color) in enumerate(
    zip(score_axes, score_columns, poster_colors),
    start=1
):
    ordered_scores = np.sort(segment_scores[score_column].to_numpy())[::-1]
    ranks = np.arange(1, len(ordered_scores) + 1)
    percentile_99 = float(np.quantile(ordered_scores, 0.99))
    axis.plot(ranks, ordered_scores, color=color, linewidth=4.5)
    axis.axhline(percentile_99, color="#D9664B", linewidth=3.5, linestyle="--")
    axis.text(
        0.97,
        0.96,
        f"99th percentile: {percentile_99:.2f}",
        transform=axis.transAxes,
        ha="right",
        va="top",
        fontsize=48,
        color="#A94736",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.9, "pad": 10}
    )
    axis.set_title(f"MDS dimension k = {k}", fontsize=64, pad=30)
    axis.set_xlabel("Segment rank", fontsize=56, labelpad=22)
    axis.set_ylabel("Mahalanobis distance", fontsize=56, labelpad=22)
    axis.tick_params(axis="both", labelsize=48, length=13, width=2.5)
    axis.grid(color="#D8DEE3", linewidth=1.5, alpha=0.8)
    axis.spines[["top", "right"]].set_visible(False)

score_export.suptitle(
    "DTEC Segment Anomaly Scores by MDS Dimension",
    fontsize=84,
    fontweight="bold"
)
score_export_png = poster_output / "dtec_mahalanobis_scores.png"
score_export_svg = poster_output / "dtec_mahalanobis_scores.svg"
score_export.savefig(score_export_png, dpi=180, facecolor="white")
score_export.savefig(score_export_svg, facecolor="white")
plt.close(score_export)

trace_export, trace_axes = plt.subplots(
    1,
    3,
    figsize=(poster_width_inches, 32)
)
for k, axis in enumerate(trace_axes, start=1):
    traces = trace_figure.data[(k - 1) * 10:k * 10]
    for trace in traces:
        axis.plot(
            trace.x,
            trace.y,
            color=trace.line.color,
            linewidth=3.2,
            alpha=0.9
        )
    axis.set_title(f"MDS dimension k = {k}", fontsize=64, pad=30)
    axis.set_xlabel("Epoch", fontsize=56, labelpad=22)
    axis.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=7))
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
    axis.tick_params(axis="both", labelsize=44, length=13, width=2.5)
    axis.grid(color="#D8DEE3", linewidth=1.5, alpha=0.8)
    axis.spines[["top", "right"]].set_visible(False)

trace_export.supylabel("DTEC", fontsize=56, fontweight="bold", x=0.015)
trace_export.suptitle(
    "Raw DTEC Traces of Top 10 Segments by MDS Dimension",
    fontsize=84,
    fontweight="bold"
)
trace_export.subplots_adjust(left=0.12, right=0.98, bottom=0.12, top=0.84, wspace=0.20)
trace_export_png = poster_output / "dtec_top_segments_traces.png"
trace_export_svg = poster_output / "dtec_top_segments_traces.svg"
trace_export.savefig(trace_export_png, dpi=180, facecolor="white")
trace_export.savefig(trace_export_svg, facecolor="white")
plt.close(trace_export)

exported_images = [score_export_png, score_export_svg, trace_export_png, trace_export_svg]
assert all(image.exists() and image.stat().st_size > 0 for image in exported_images)
for image in exported_images:
    print(f"{image}: {image.stat().st_size / 1_000_000:.1f} MB")

output/poster/dtec_mahalanobis_scores.png: 0.5 MB
output/poster/dtec_mahalanobis_scores.svg: 0.1 MB
output/poster/dtec_top_segments_traces.png: 0.9 MB
output/poster/dtec_top_segments_traces.svg: 0.5 MB
